# Lab 0: Qiskit Global Summer School 2026에 오신 것을 환영합니다

Lab 0은 본편에 들어가기 전 몸을 푸는 노트북입니다. 여기서는 다음을 해봅니다.

- Python 패키지 업그레이드
- Sampler와 Estimator로 간단한 코딩 연습 풀기
- 특별한 다중 큐비트 양자 회로를 만들고 그 상태벡터 들여다보기
- *(선택)* Qiskit C API로 C에서 양자 회로 만들기

기대되나요? 앞으로 다룰 내용이 훨씬 많습니다.


이 랩은 macOS, Linux, 그리고 [qBraid](https://qbraid.com/)나 [Google Colab](https://colab.research.google.com/) 같은 클라우드 플랫폼에서 돌아갑니다. Windows를 쓴다면 클라우드 플랫폼을 권합니다. 앞으로의 랩에서 일부 패키지는 Linux 기반 환경에서 가장 잘 동작하기 때문입니다.

정리하면 이렇습니다.

| 사용 중인 OS | 실행 방법 |
|---|---|
| macOS / Linux | 로컬에서 실행하세요. 설정은 노트북이 알아서 처리합니다. |
| Windows | 1~2장은 Python과 venv로 로컬에서 실행됩니다. 선택 사항인 3장은 [qBraid](https://qbraid.com/)나 [Google Colab](https://colab.research.google.com/)에서 이 노트북을 여세요. |

<br>


## 설치

먼저 이 셀을 실행해 Python 버전을 확인하세요.

In [ ]:
import sys

recommended = (3, 10)
current = sys.version_info[:2]

if current < recommended:
    print(f"현재 Python {current[0]}.{current[1]}을(를) 사용 중입니다. Python {recommended[0]}.{recommended[1]} 이상을 권장합니다.")
else:
    print(f"Python {current[0]}.{current[1]} — 좋습니다!")


이제 아래 셀을 실행해 필요한 패키지를 설치하고 업그레이드합니다. 업그레이드 중에 로컬에 이미 깔린 패키지와 의존성이 충돌하면, __새 Python 환경__(Python 3.12 권장)을 만들어 거기서 챌린지를 이어가세요.

환경을 깨끗하게 새로 잡고 Qiskit을 설치하는 방법은 IBM Quantum 문서에 정리돼 있습니다: https://quantum.cloud.ibm.com/docs/guides/install-qiskit

In [ ]:
# 필요한 패키지를 설치합니다.
# 이 노트북을 처음 실행할 때는 아래 줄의 주석을 풀어 의존성을 설치하고,
# 이후 실행에서는 다시 주석 처리하세요.


%pip install --upgrade 'qiskit[visualization]>=2.5.0' qiskit-aer qiskit-ibm-runtime matplotlib
%pip install --upgrade qc-grader

설치와 업그레이드가 끝나면 Python 커널을 재시작하고, 아래 라이브러리 임포트 셀부터 노트북을 다시 실행하세요.

In [ ]:
import os
import platform
import shutil
import subprocess
import sys

import qc_grader
import qiskit

from qc_grader.challenges.qgss_2026 import check_progress
from qc_grader.challenges.qgss_2026.lab0 import (
    grade_lab0_ex1,
    grade_lab0_ex2,
    grade_lab0_ex3,
    grade_lab0_ex4,
)


from qiskit.circuit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit.visualization import array_to_latex, plot_histogram, plot_state_qsphere
from qiskit_aer import AerSimulator
from qiskit_ibm_runtime import Batch, QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator
from qiskit_ibm_runtime import SamplerV2 as Sampler

import matplotlib.pyplot as plt

print(f"Qiskit version: {qiskit.__version__}")
print(f"Grader version: {qc_grader.__version__}")


# 1장 — Qiskit 패턴으로 만드는 첫 양자 프로그램

## 1.1 IBM Quantum 계정

먼저 할 일이 있습니다. QGSS의 랩들에 도전하려면 활성화된 IBM Quantum 계정이 필요합니다.

이미 `QiskitRuntimeService`로 인증 정보를 저장해 뒀다면 아래 셀만 실행하면 백엔드 목록이 나타납니다. 처음이거나 설정한 지 오래됐다면 다음 순서로 계정을 저장하세요.

1. [quantum.cloud.ibm.com](https://quantum.cloud.ibm.com/)에 접속해 로그인하세요. 계정이 없으면 새로 만들면 됩니다.
2. 대시보드에서 `Create +`나 `View all`을 눌러 API 키를 복사하세요.
3. Instances 탭에서 인스턴스 CRN을 복사하세요.
4. 아래 셀의 자리표시자를 본인 API 키와 인스턴스 CRN으로 바꾼 뒤 실행하세요.

끝나면 grader 셀을 실행해 인증 정보가 제대로 설정됐는지 확인하세요.

In [ ]:
# 여기에 IBM Cloud 인증 정보를 붙여넣으세요. 환경마다 한 번만 하면 됩니다.
# 정상 동작이 확인되면, 인증 정보가 파일에 남지 않도록 이 줄들을 주석 처리해도 됩니다.

#token = "<YOUR_API_KEY>"
#instance = "<YOUR_CRN>"  # "crn:v1:bluemix:public:quantum-computing:..." 같은 형식입니다

#QiskitRuntimeService.save_account(
#    token=token,
#    instance=instance,
#    overwrite=True,
#    set_as_default=True,
#)

# 확인
service = QiskitRuntimeService()
backends = service.backends()
print(f"계정 정상. 사용 가능한 백엔드 {len(backends)}개:")
for b in backends[:5]:
    print(f"  {b.name} ({b.num_qubits} qubits)")


<div class="alert alert-block alert-success">
  <b>연습 1: IBM Quantum 계정 확인</b>

  API 키와 CRN을 설정하고 사용 가능한 백엔드를 확인했나요?
  아래 연습 1 셀을 실행해 grader가 잘 동작하는지 확인하세요.
  따로 입력할 건 없습니다.

  계정이 제대로 설정됐다면 성공 메시지가 나타납니다.
</div>


In [ ]:
grade_lab0_ex1()

## 1.2 게이트로 하는 양자 계산

IBM 양자 컴퓨터는 게이트 기반입니다. 고전 컴퓨터가 AND, OR, NOT 게이트로 비트를 처리하듯, 양자에서는 양자 게이트를 조합해 알고리즘을 만듭니다. 큰 차이는 양자 게이트가 고전 비트의 양자 버전인 큐비트에 작용한다는 점입니다. 고전 비트는 0 아니면 1이지만, 큐비트는 세 가지 양자역학적 현상을 보입니다. 첫째는 중첩으로, 큐비트가 측정 전까지 0과 1의 선형 결합으로 존재합니다. 둘째는 얽힘으로, 둘 이상의 큐비트가 각자만으로는 기술할 수 없는 공동의 양자 상태를 공유합니다. 셋째는 간섭으로, 양자 진폭이 서로 보강되거나 상쇄되어 알고리즘이 틀린 답을 억누르고 맞는 답을 키울 수 있습니다. 바로 이 세 현상이 양자 계산을 고전 계산과 차별화 시킵니다.

### 중첩과 하다마드 게이트

[하다마드 게이트 (H)](https://quantum.cloud.ibm.com/docs/api/qiskit/qiskit.circuit.library.HGate)는 $|0\rangle$이나 $|1\rangle$처럼 확정된 상태의 큐비트를 두 상태가 똑같이 섞인 중첩으로 만듭니다.

$$H = \frac{1}{\sqrt{2}}\begin{pmatrix} 1 & 1 \\ 1 & -1 \end{pmatrix}, \qquad H|0\rangle = \frac{1}{\sqrt{2}}\bigl(|0\rangle + |1\rangle\bigr)$$

이 상태를 측정하면 0과 1이 각각 50% 확률로 나옵니다. $\frac{1}{\sqrt{2}}$ 계수는 확률 진폭이고, 각 결과가 측정될 확률은 그 진폭의 제곱입니다.

아래 셀을 완성해 하다마드 게이트가 든 첫 1큐비트 회로를 만들어 봅시다.

<br>

<div class="alert alert-block alert-info">

중첩, 얽힘, 간섭을 더 알고 싶다면 [양자 정보의 기초](https://quantum.cloud.ibm.com/learning/courses/basics-of-quantum-information) 과정을 보세요.

</div>


<div class="alert alert-block alert-success">
<b>연습 2: 첫 양자 회로 만들기</b>

아래는 비어 있는 1큐비트 회로입니다. 큐비트 0에 <b>하다마드 게이트</b>를 추가해 중첩 상태로 만드세요. 1.6절에서 Sampler를 돌릴 때 나오는 50/50 분포가 바로 여기서 비롯됩니다.

힌트: 공식 [퀵스타트](https://quantum.cloud.ibm.com/docs/guides/quick-start)를 참고하세요.

</div>


In [ ]:

qc = QuantumCircuit(1)

# TODO: 큐비트 0에 하다마드 게이트를 추가하세요

qc.draw("mpl")


## 1.3 Qiskit 패턴이란?

이제 $H|0\rangle$ 중첩 상태를 준비하는 회로가 생겼습니다. 다음 절들에서는 이 회로로 두 가지 실험을 합니다. 하나는 측정 카운트를 모으고, 다른 하나는 Z 관측가능량의 기댓값을 계산합니다. 두 실험 모두 [Qiskit 패턴](https://quantum.cloud.ibm.com/docs/guides/intro-to-patterns)이라 부르는 같은 4단계 흐름을 따릅니다.

| 단계 | 하는 일 | 이 랩에서는 |
|------|------------|-------------|
| 1. 맵 (Map) | 문제를 양자 회로와 연산자로 옮긴다. | H 회로를 만들고 Z 관측가능량을 정의합니다. |
| 2. 최적화 (Optimize) | 대상 하드웨어에 맞게 회로를 다듬는다. | `generate_preset_pass_manager`로 처리합니다. |
| 3. 실행 (Execute) | 프리미티브로 하드웨어에서 실행한다. | Sampler와 Estimator 둘 다로 회로를 실행합니다. |
| 4. 후처리 (Post-process) | 결과를 후처리한다. | 측정 히스토그램을 그리고 ⟨Z⟩ ≈ 0을 확인합니다. |

실행 단계에서 Qiskit은 두 가지 프리미티브를 제공합니다.

- [SamplerV2](https://quantum.cloud.ibm.com/docs/guides/primitives)는 회로 실행 결과의 출력 레지스터를 샘플링합니다. 이걸로 측정 카운트를 모아 0과 1이 대략 같은 빈도로 나오는지 봅니다.
- [EstimatorV2](https://quantum.cloud.ibm.com/docs/guides/primitives)는 회로가 준비한 상태에 대해 관측가능량의 기댓값을 계산합니다. 이걸로 $\langle Z \rangle$를 구합니다. $|0\rangle$의 고윳값은 $+1$, $|1\rangle$의 고윳값은 $-1$이므로, 균등한 중첩에서는 $\langle Z \rangle = 0$이 됩니다.

다음 절들에서 각 단계를 차례로 살펴봅니다.

## 1.4 1단계 — 맵 (Map)

맵 단계에서는 양자 문제를 회로와 관측가능량으로 정의합니다.

회로는 이미 준비돼 있습니다. `qc`가 $H|0\rangle$ 상태를 만듭니다. 여기서는 관측가능량을 정합니다. $|0\rangle$의 고윳값이 $+1$, $|1\rangle$의 고윳값이 $-1$인 Z 연산자(파울리 Z)를 씁니다.

$$Z = |0\rangle\langle 0| - |1\rangle\langle 1|$$

$H|0\rangle$ 상태에서는 두 결과가 똑같이 나오므로 기댓값은 다음과 같습니다.

$$\langle Z \rangle = \frac{1}{2}(+1) + \frac{1}{2}(-1) = 0$$

이 값을 실행 단계에서 Estimator로 확인합니다.

<div class="alert alert-block alert-success">
<b>연습 3: 관측가능량 정의하기</b>

맵 단계에서는 무엇을 측정할지 정합니다. 여기서 필요한 건 큐비트 0의 파울리 Z 연산자입니다. 그 기댓값은 큐비트가 |0⟩과 |1⟩ 중 어느 쪽으로 얼마나 치우쳤는지 알려줍니다.

계수가 $1.0$인 단일 큐비트 Z __관측가능량__을 <code>SparsePauliOp</code>로 만드세요.

힌트: [SparsePauliOp 문서](https://quantum.cloud.ibm.com/docs/api/qiskit/qiskit.quantum_info.SparsePauliOp)에 참고할 예제가 있습니다.

</div>


In [ ]:

# TODO: 관측가능량을 단일 큐비트 Z 연산자로 정의하세요
observable = None

print(f"Observable: {observable}")


## 1.5 2단계 — 트랜스파일 (Transpile)

양자 하드웨어에는 제약이 있습니다. 실제 하드웨어는 밑바탕의 물리적 구현이 정하는 네이티브 게이트 집합과 큐비트 연결 구조를 가집니다. 트랜스파일 단계는 추상 회로를, 대상 백엔드가 실제로 실행할 수 있는 동등한 회로로 다시 씁니다. 이때 백엔드가 지원하는 게이트와 연결 구조만 사용합니다.

이 작업은 `generate_preset_pass_manager`가 맡습니다. 아래 셀은 거의 모든 게이트를 받아들이는 이상적인 시뮬레이터를 대상으로 하므로, 트랜스파일된 회로가 원본과 크게 다르지 않습니다. 선택 사항인 1.8절에서는 실제 하드웨어를 대상으로 하는데, 이때는 트랜스파일러가 할 일이 더 많습니다. 게이트를 백엔드의 네이티브 게이트로 분해하고, 연결 제약에 맞춰 회로 배치를 최적화합니다.

트랜스파일은 회로와 관측가능량에 함께 적용됩니다. 트랜스파일러가 가상 큐비트를 물리 큐비트에 배정하면, 관측가능량도 같은 배정을 따라야 합니다. 이건 `apply_layout`이 처리합니다. 지금은 큐비트가 하나뿐이라 배치가 뻔하지만, 실제 하드웨어의 다중 큐비트 회로에서도 똑같은 방식이 필요합니다.


In [ ]:
backend = AerSimulator()
pm = generate_preset_pass_manager(optimization_level=1, backend=backend)

# Sampler는 회로 안에 측정이 있어야 합니다
qc_meas = qc.copy()
qc_meas.measure_all()
isa_qc_meas = pm.run(qc_meas)

# Estimator는 측정이 없는 회로를 받아 관측가능량을 별도로 평가합니다
isa_qc = pm.run(qc)

# 트랜스파일된 회로와 큐비트 인덱스가 맞도록 관측가능량에 동일한 배치를 적용합니다
isa_observable = observable.apply_layout(isa_qc.layout)

print("트랜스파일된 회로와 관측가능량이 준비됐습니다.")

## 1.6 3단계 — 실행 (Execute)

이제 트랜스파일된 회로를 두 프리미티브로 실행합니다. 둘은 같은 양자 상태를 두고 서로 다른 질문에 답합니다.

### Sampler: "어떤 카운트가 나올까?"

Sampler는 회로를 여러 번(샷) 돌려 각 결과가 얼마나 자주 나오는지 셉니다. 0이 몇 번, 1이 몇 번 측정되는지를 보는 것이죠.

In [ ]:
sampler = Sampler(mode=backend)
sampler_result = sampler.run([isa_qc_meas], shots=1000).result()
counts = sampler_result[0].data.meas.get_counts()

print("Sampler counts:", counts)

<div class="alert alert-block alert-success">
<b>Sampler 결과를 제출해 연습 2 완료하기</b>

방금 구한 <code>counts</code>는 1.2절에서 만든 H 회로에서 나온 값입니다. 아래 셀을 실행해 제출하세요. grader가 50/50 분포인지 확인합니다.

</div>


In [ ]:
grade_lab0_ex2(counts)

### Estimator: "회로를 관측하면 어떤 값이 나올까?"

Estimator는 회로와 관측가능량을 받아 기댓값을 바로 돌려줍니다. 카운트를 일일이 모아 평균 낼 필요가 없습니다. 회로에 측정 게이트도 필요 없고, 측정은 Estimator가 내부에서 처리합니다.

$H|0\rangle$ 상태라면 $\langle Z \rangle = 0$이 나와야 합니다.


In [ ]:
# Estimator 실행 — 측정 없는 회로 + 관측가능량
estimator = Estimator(mode=backend)
estimator_result = estimator.run([(isa_qc, isa_observable)]).result()
exp_val = estimator_result[0].data.evs

print(f"⟨Z⟩ = {exp_val:.4f}")

<div class="alert alert-block alert-success">
<b>Estimator 결과를 제출해 연습 3 완료하기</b>

아래 셀을 실행해 <code>exp_val</code>을 grader에 제출하세요. 완벽한 H|0⟩ 중첩이라면 ⟨Z⟩는 ≈ 0이어야 합니다.

</div>


In [ ]:
grade_lab0_ex3(exp_val)

## 1.7 4단계 — 후처리 (Post-process)

후처리 단계에서는 원시 결과에서 의미를 뽑아냅니다. 문제에 따라 측정 분포를 그리기도 하고, 기댓값에서 다른 값을 계산하기도 하고, 결과를 고전 최적화 루프에 넘기거나 오류 완화 기법을 적용하기도 합니다.

이 랩에서는 간단하게 갑니다. Sampler 카운트를 히스토그램으로 그려 0과 1이 몇 번 측정됐는지 보고, Estimator 결과는 $\langle Z \rangle$의 통계적 불확실성을 오차 막대로 함께 표시한 막대그래프로 그립니다. 시뮬레이터에서는 카운트가 고르게 나뉘고 $\langle Z \rangle \approx 0$이 됩니다. 실제 하드웨어에서는 두 그래프 모두 샷 노이즈와 장치 노이즈의 영향을 담습니다.

In [ ]:

# Sampler 결과
print(f"Sampler counts: {counts}")
display(plot_histogram(counts, title="H|0⟩ measurement distribution (1000 shots)"))

# Estimator 결과
ev = estimator_result[0].data.evs.item()
std = estimator_result[0].data.stds.item()
print(f"\nEstimator ⟨Z⟩ = {ev:.4f} ± {std:.4f}")

fig, ax = plt.subplots(figsize=(4, 5))
ax.bar(["⟨Z⟩"], [ev], yerr=[std], capsize=10, color="steelblue", width=0.4)
ax.axhline(0, color="gray", linewidth=0.8, linestyle="--")
ax.set_ylim(-1.2, 1.2)
ax.set_ylabel("Expectation value")
ax.set_title("Z observable on H|0⟩ (Estimator)")
plt.tight_layout()
plt.show()


## 1.8 (선택) 실제 양자 하드웨어에서 실행하기

<div class="alert alert-block alert-info">
<b>이 절은 선택 사항이고 채점하지 않습니다.</b> 실제 양자 컴퓨터에 작업을 보내며 QPU 시간을 씁니다. 건너뛰고 2장으로 가도 됩니다.
</div>

지금까지는 시뮬레이터를 썼습니다. 이제 같은 회로를 실제 IBM 양자 프로세서에서 돌립니다. 코드는 앞과 같은 Qiskit 패턴 흐름을 따르고, 바뀌는 건 백엔드뿐입니다.

실제 하드웨어는 시뮬레이터에 없는 물리적 제약을 받습니다. 게이트는 물리 큐비트에 가하는 마이크로파 펄스로 실행되고, 연산마다 충실도가 완벽하지 않습니다. 그래서 카운트는 50/50에 가깝지만 딱 맞지는 않고, $\langle Z \rangle$도 0 근처에서 측정 가능한 불확실성을 안고 나옵니다. 이 결과를 시뮬레이터와 견줘 보면 양자 오류 완화가 어디서 필요해지는지 감이 옵니다.

Sampler와 Estimator 작업은 [Batch](https://quantum.cloud.ibm.com/docs/guides/run-jobs-batch)로 묶어 함께 보냅니다. Batch는 여러 작업을 한 번의 백엔드 예약으로 묶어, 다시 줄 서지 않고 연달아 실행되게 합니다. 이 작업은 `Heron` 장치에서 QPU 시간을 약 20초 씁니다. 대기열 대기 시간은 백엔드 부하에 따라 달라집니다. 작업 ID가 찍히기 전에 셀이 타임아웃되면, 다음 셀에서 작업 ID로 결과를 가져오는 방법을 보여줍니다.

In [ ]:
# 기본적으로 이 절은 실제 양자 하드웨어에서 실행됩니다.
# 본편을 위해 QPU 시간을 아끼려면, 아래 노이즈 시뮬레이터 줄의 주석을 풀고
# 이 절의 나머지에서 real_backend를 real_backend_sim으로 바꾸세요.

real_backend = service.least_busy()
# real_backend_sim = AerSimulator.from_backend(real_backend)  # 노이즈 시뮬레이터

print(f"Running on: {real_backend.name} ({real_backend.num_qubits} qubits)")

# 2단계: 대상 백엔드에 맞게 트랜스파일
pm_real = generate_preset_pass_manager(optimization_level=1, backend=real_backend)

qc_meas = qc.copy()
qc_meas.measure_all()
isa_qc_meas = pm_real.run(qc_meas)

isa_qc_real = pm_real.run(qc)
isa_obs = observable.apply_layout(isa_qc_real.layout)

# 3단계: 실행 — 두 프리미티브를 하나의 Batch에서 실행합니다
with Batch(backend=real_backend):
    sampler = Sampler()
    sampler.options.environment.job_tags = ["qgss26"]
    sampler_job = sampler.run([isa_qc_meas], shots=1000)

    estimator = Estimator()
    estimator.options.environment.job_tags = ["qgss26"]
    estimator_job = estimator.run([(isa_qc_real, isa_obs)])

sampler_job_id = sampler_job.job_id()
estimator_job_id = estimator_job.job_id()

print(f"Sampler job ID:   {sampler_job_id}")
print(f"Estimator job ID: {estimator_job_id}")


In [ ]:
# (선택) 위 셀이 결과를 출력하기 전에 타임아웃됐다면, 그 출력에서
# Sampler/Estimator 작업 ID를 아래 문자열에 복사하고 이 셀을 실행하세요.
# 그렇지 않으면 주석 처리된 상태로 두세요.

# sampler_job_id = "<paste sampler job id>"
# estimator_job_id = "<paste estimator job id>"
# sampler_job = service.job(sampler_job_id)
# estimator_job = service.job(estimator_job_id)


In [ ]:
# 4단계: 결과
hw_counts = sampler_job.result()[0].data.meas.get_counts()
hw_estimator_result = estimator_job.result()
hw_exp_val = hw_estimator_result[0].data.evs

print(f"Hardware Sampler counts: {hw_counts}")
print(f"Hardware Estimator ⟨Z⟩:  {hw_exp_val:.4f}")

결과를 나란히 놓고 비교해 봅시다. 시뮬레이터는 50/50으로 나뉩니다. 하드웨어 결과는 노이즈 때문에 거기서 조금씩 어긋납니다.

In [ ]:
display(plot_histogram(
    [counts, hw_counts],
    legend=["Simulator", f"Hardware ({real_backend.name})"],
    title="H|0⟩ — Simulator vs Real Hardware",
))


# Estimator 결과
ev = hw_estimator_result[0].data.evs.item()
std = hw_estimator_result[0].data.stds.item()
print(f"\nEstimator ⟨Z⟩ = {ev:.4f} ± {std:.4f}")

fig, ax = plt.subplots(figsize=(4, 5))
ax.bar(["⟨Z⟩"], [ev], yerr=[std], capsize=10, color="steelblue", width=0.4)
ax.axhline(0, color="gray", linewidth=0.8, linestyle="--")
ax.set_ylim(-1.2, 1.2)
ax.set_ylabel(f"Expectation value from ({real_backend.name})")
ax.set_title("Z observable on H|0⟩ (Estimator)")
plt.tight_layout()
plt.show()


# 2장 — 다중 큐비트 회로와 시각화

1장에서 여러분은 단일 큐비트 중첩을 만들고, Sampler와 Estimator를 사용해서 중첩의 양자 상태가 두 측정 관점에서 어떻게 보이는지 확인했습니다. 시뮬레이터에서, 그리고 시도했다면 실제 하드웨어에서도요.

이제 좀 더 흥미로운 걸 만들어 봅시다. 이 장에서는 큐비트 6개로 특별한 회로를 한 단계씩 쌓습니다. 마지막에는 회로가 만든 상태벡터를 [QSphere](https://quantum.cloud.ibm.com/docs/api/qiskit/qiskit.visualization.plot_state_qsphere)로 시각화합니다. QSphere는 각 기저 상태의 확률과 위상을 보여주는 다중 큐비트 시각화 도구입니다. 최종 상태가 무엇을 닮았는지 눈치채게 될 수도 있어요!

## 2.1 6큐비트 얽힘 상태 만들기

이 연습에서는 게이트 세 개를 씁니다.

- [H](https://quantum.cloud.ibm.com/docs/api/qiskit/qiskit.circuit.library.HGate): 큐비트를 $|0\rangle$과 $|1\rangle$의 균등한 중첩으로 만듭니다. 이미 아는 게이트죠.
- [X](https://quantum.cloud.ibm.com/docs/api/qiskit/qiskit.circuit.library.XGate): 큐비트를 $|0\rangle$에서 $|1\rangle$로, 또는 그 반대로 뒤집습니다. 고전 NOT 게이트의 양자 버전입니다.
- [CX](https://quantum.cloud.ibm.com/docs/api/qiskit/qiskit.circuit.library.CXGate): 2큐비트 게이트로, 제어 큐비트가 $|1\rangle$일 때만 대상 큐비트를 뒤집습니다. 중첩 상태에 CX를 걸면 두 큐비트가 얽힙니다.

<div class="alert alert-block alert-info">

이 게이트들을 더 파고들고 실제 하드웨어에서 돌려보고 싶다면 <a href="https://quantum.cloud.ibm.com/learning/courses/use-a-qc-today/build-and-run-your-first-quantum-program">Build and run your first quantum program</a>을 보세요.

</div>


모든 큐비트는 $|0\rangle$에서 시작합니다. 아래 셀부터 시작하세요.


<div class="alert alert-block alert-success">
<b>연습 4: 6큐비트 얽힘 상태 만들기</b>

6큐비트 회로를 만듭니다. 나중에 QSphere로 시각화하면 익숙한 모양이 보일 겁니다.

아래 셀들로 세 단계에 걸쳐 만듭니다.

1. (이미 주어짐) 6큐비트 회로를 만듭니다.
2. 큐비트 0에 <b>H</b>를 걸어 중첩으로 만든 뒤, 큐비트 1에 <b>X</b>를 겁니다.
3. 큐비트 0을 제어로, 큐비트 1부터 5까지를 대상으로 <b>CNOT 체인</b>을 걸어 모두 얽히게 합니다.

힌트: `for` 루프로 CNOT 체인을 만들 수 있습니다.
</div>

In [ ]:
# 1단계: 6큐비트 회로 만들기
special_qc = QuantumCircuit(6)
special_qc.draw("mpl")

이제 큐비트 0에 H를 걸어 중첩으로 만듭니다. 이어 큐비트 1에 X를 걸어 $|0\rangle$에서 $|1\rangle$로 뒤집습니다. 이제 두 큐비트가 각각 원하는 상태로 준비됐고, 다음 단계에서 얽힐 채비가 끝났습니다.

In [ ]:
# 2단계: 큐비트 0에 H, 큐비트 1에 X
# TODO: 큐비트 0에 H를 적용하세요

# TODO: 큐비트 1에 X를 적용하세요


special_qc.draw('mpl')


이제 CNOT 체인으로 얽힘을 만듭니다. 각 CNOT은 큐비트 0을 제어로 삼고 큐비트 1부터 5까지를 대상으로 합니다. 이렇게 하면 모든 큐비트가 하나로 이어집니다. 큐비트 0이 중첩에 있으니, 나머지 큐비트도 큐비트 0과 얽힙니다. 회로도에는 큐비트 0을 나머지 큐비트에 잇는 CNOT 게이트 다섯 개가 보입니다.


In [ ]:
# 3단계: CNOT 체인 — 모든 큐비트를 큐비트 0과 얽히게 합니다
# TODO: 큐비트 1부터 5까지 반복하며 큐비트 0을 제어로 하는 CNOT을 적용하세요


special_qc.draw('mpl')


## 2.2 상태벡터

상태벡터는 양자 상태를 빠짐없이 적은 수학적 기술입니다. 큐비트가 $n$개면 상태벡터는 $2^n$개의 진폭으로 이뤄진 복소 벡터이고, 진폭 하나가 기저 상태 하나에 대응합니다. 특정 기저 상태가 측정될 확률은 그 진폭의 제곱입니다.

6큐비트 회로에서는 상태벡터에 항목이 $2^6 = 64$개 있고, $|000000\rangle$부터 $|111111\rangle$까지 모든 기저 상태를 담습니다. `Statevector` 클래스가 회로에서 이 값을 계산하고, `array_to_latex`가 수식 형태로 출력합니다.

항목 대부분은 0입니다. 진폭이 0이 아닌 기저 상태만 여러분이 만든 양자 상태에 기여합니다.

In [ ]:
sv = Statevector(special_qc)
array_to_latex(sv, max_size=64)

## 2.3 양자 상태 시각화하기

복소수 64개를 그대로 읽는 건 불편하고 직관적이지 않습니다. 구조를 한눈에 볼 방법이 필요합니다.

큐비트가 하나라면 [블로흐 구](https://quantum.cloud.ibm.com/docs/api/qiskit/qiskit.visualization.plot_bloch_multivector)가 어떤 순수 상태든 구면 위의 점으로 나타냅니다. 하지만 블로흐 구는 한 번에 큐비트 하나만 다루고, 블로흐 구 여섯 개를 따로 그려도 큐비트가 어떻게 얽혔는지는 드러나지 않습니다.

[QSphere](https://quantum.cloud.ibm.com/docs/api/qiskit/qiskit.visualization.plot_state_qsphere)는 다중 큐비트 상태 전체를 구면 하나에 담습니다. 각 기저 상태는 구면 위의 점이고, 점마다 세 가지 정보를 담습니다.

- 위도는 해밍 무게, 곧 비트열에 든 1의 개수로 정해집니다. 1이 없는 상태는 북극에, 전부 1인 상태는 남극에 놓입니다.
- 점 크기는 그 기저 상태가 측정될 확률을 나타냅니다. 진폭이 0인 상태는 나타나지 않습니다.
- 색은 양자 위상을 나타냅니다.

여러분이 만든 상태에서는 크기가 같은 점 두 개가 보입니다. 두 점이 어떤 기저 상태에 대응하는지 살펴보세요.

In [ ]:
# QSphere
plot_state_qsphere(sv)

### QSphere가 보여주는 것

QSphere에 점이 두 개 있습니다. 하나는 위쪽에, 하나는 아래쪽에요. 얽힘으로 이어진 채 균등한 중첩을 이루는 두 기저 상태입니다.

<div class="alert alert-block alert-success">
<b>연습 4 — 상태벡터 제출하기</b>

<code>sv</code>를 grader에 제출해 연습 4를 완료하세요. grader가 회로를 확인하고, 이 상태가 왜 특별한지 알려줍니다. 본편 준비에 도움이 될 과정 링크도 함께 받게 됩니다. 아래 셀을 실행해 Lab 0을 마무리하세요.

</div>


In [ ]:
grade_lab0_ex4(sv)


### 다음은?

IBM은 2016년 5월 4일에 첫 양자 컴퓨터를 클라우드에 올렸습니다. 그 뒤로 Qiskit과 양자 컴퓨팅 커뮤니티는 하드웨어, 소프트웨어, 교육의 세대를 거듭하며 함께 자라 왔습니다. Qiskit은 커뮤니티가 함께 만든 도구이고, 여러분의 양자 여정 내내 함께합니다.

이 랩에서 여러분은 시뮬레이터와 실제 하드웨어에서 회로를 돌리고, 서로 다른 두 프리미티브로 양자 상태를 측정하고, 6큐비트 얽힘 상태를 시각화했습니다. 모두 QGSS 2026 내내 쓰게 될 기본기입니다.

진행 상황을 확인하며 주요 연습을 마무리합시다. 3장에는 Qiskit을 쓰는 또 다른 방법, 곧 C로 양자 회로를 만드는 선택 연습이 있습니다. 준비되면 들러 주세요.


In [ ]:
check_progress("lab0")

# 3장 (선택, 채점 안 함) — C에서 양자 회로 만들기

<div class="alert alert-block alert-info">
<b>이 장은 선택 사항이고 채점하지 않습니다.</b> C 컴파일러(GCC 또는 Clang)가 필요합니다. Windows 사용자는 시작하기 전에 3.1절을 보세요.
</div>

앞 장들에서는 Python으로 회로를 만들었습니다. Qiskit은 [C API](https://www.ibm.com/quantum/blog/c-api-enables-end-to-end-hpc-demo)도 제공해, C에서 회로를 만들어 Python으로 넘길 수 있습니다. 양자 프로그램이 컴파일 언어로 짠 고전 고성능 컴퓨팅(HPC) 작업과 나란히 돌아갈 때 유용합니다.

이 장은 [Extend Qiskit in Python with C](https://quantum.cloud.ibm.com/docs/guides/c-extension-for-python) 가이드를 따릅니다. 2장의 회로를 C로 다시 만든 뒤, Python으로 상태벡터를 계산합니다.

<div class="alert alert-block alert-warning">
Qiskit C API는 실험적이라, 빌드에 쓰는 버전이 설치된 런타임의 마이너 버전과 맞아야 합니다. 이 노트북은 최신 Qiskit을 설치하므로 빌드와 런타임이 늘 같은 버전입니다. 자세한 내용은 IBM의 <a href="https://quantum.cloud.ibm.com/docs/guides/c-extension-for-python">C 확장 가이드</a>를 보세요.
</div>


## 3.1 설정

Qiskit C API 헤더와 라이브러리는 pip로 Qiskit을 설치할 때 함께 들어오므로 따로 받을 필요가 없습니다. C 확장을 빌드하려면 C 컴파일러와 `setuptools`가 필요합니다.

macOS, Linux, Google Colab, qBraid에서는 별도 설정 없이 됩니다. 아래 설정 셀을 실행해 시작하세요.

<details>
<summary><b>Windows 사용자에게</b></summary>

<br>

3장은 Windows를 공식 지원하지 않습니다. Windows는 컴파일러 툴체인, MSVC 버전, BLAS 유무, 경로 처리가 환경마다 크게 달라, 노트북 하나로 모든 구성에서 매끄러운 빌드를 보장하기 어렵습니다.

번거로움 없이 진행하려면 C 컴파일러가 이미 갖춰진 <a href="https://qbraid.com/">qBraid</a>나 <a href="https://colab.research.google.com/">Google Colab</a>에서 이 노트북을 여세요.

</details>

<div class="alert alert-block alert-info">
이미 GCC나 Clang이 있고 직접 설정하고 싶다면, 설정 셀을 건너뛰고 IBM 문서를 그대로 따라도 됩니다.

<ul>
<li><a href="https://quantum.cloud.ibm.com/docs/guides/install-c-api">Install the Qiskit C API</a></li>
<li><a href="https://quantum.cloud.ibm.com/docs/guides/c-extension-for-python">Extend Qiskit in Python with C</a></li>
<li><a href="https://quantum.cloud.ibm.com/docs/api/qiskit-c">Qiskit C API reference</a></li>
</ul>
</div>

In [ ]:
import sys
import os
import platform
import shutil
import subprocess

from qiskit.quantum_info import Statevector
from qiskit.visualization import array_to_latex, plot_state_qsphere

# macOS / Linux / Colab / qBraid용 설정.
# Windows에서는 3.1절의 안내를 참고해 qBraid나 Google Colab을 사용하세요.

if platform.system() == "Windows":
    print("Windows: 이 장은 qBraid나 Google Colab에서 실행하세요.")
else:
    IS_COLAB = "google.colab" in sys.modules

    # Colab에는 기본적으로 컴파일러가 없습니다.
    if IS_COLAB and shutil.which("gcc") is None:
        subprocess.run(["apt-get", "-qq", "install", "-y", "build-essential"], check=True)

    # setuptools가 C 파일을 Python 모듈로 빌드합니다.
    try:
        import setuptools  # noqa: F401
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "setuptools"], check=True)

    cc = shutil.which("gcc") or shutil.which("clang")
    if cc is None:
        print("C 컴파일러를 찾을 수 없습니다. macOS에서는 다음을 실행하세요: xcode-select --install")
    else:
        print("C compiler:", cc)
        print("설정 완료. 3.2로 진행하세요.")

## 3.2 3장의 회로를 C로 만들기

이 연습에서 쓸 C API 함수:

| 함수 | 하는 일 |
|---|---|
| `qk_circuit_new(num_qubits, num_clbits)` | 새 회로를 만듭니다 |
| `qk_circuit_gate(qc, QkGate_H, qubits, NULL)` | 게이트를 추가합니다. `qubits`는 큐비트 인덱스를 담은 `uint32_t` 배열입니다. `QkGate_H`, `QkGate_X`, `QkGate_CX`를 씁니다. |
| `qk_circuit_measure(qc, qubit, clbit)` | 측정을 추가합니다 |
| `qk_circuit_to_python_full(qc)` | C 회로를 Python `QuantumCircuit`으로 바꾸고 소유권을 넘깁니다. 이 뒤에는 `qk_circuit_free`를 부르지 마세요. |

Qiskit C API를 쓰는 모든 C 확장이 지켜야 할 규칙 두 가지:

- `QISKIT_PYTHON_EXTENSION`을 정의하고, `qiskit.h`보다 먼저 `Python.h`를 인클루드하세요.
- `PyInit_<name>` 안에서 `qk_import()`를 한 번 부르세요. 이걸 빼면 API 호출이 실행 중에 실패합니다.

전체 뼈대를 갖춘 완성 예제는 [C 확장 가이드](https://quantum.cloud.ibm.com/docs/guides/c-extension-for-python)에 있습니다.

<div class="alert alert-block alert-info">
Python은 컴파일된 모듈을 세션당 한 번만 불러옵니다. C 파일을 고쳐 다시 빌드했다면, 다시 임포트하기 전에 커널을 재시작하세요.
</div>


<div class="alert alert-block alert-success">
<b>선택 연습: 다중 큐비트 회로를 C로 만들기</b>

연습 4와 같은 6큐비트 회로를 이번에는 C로 만듭니다. 아래 파일에서 게이트 세 단계를 채우세요.

1. 큐비트 0에 <b>H</b>를 겁니다.
2. 큐비트 1에 <b>X</b>를 겁니다.
3. <b>CNOT 체인</b>을 추가합니다. 제어는 큐비트 0, 대상은 큐비트 1부터 5까지입니다. <code>for</code> 루프를 쓰면 됩니다.

이 회로에는 측정이 없으므로, 다음 단계에서 상태벡터를 계산할 수 있습니다.
</div>

In [ ]:
%%writefile qgss_circuit.c

#include <Python.h>
#include <qiskit.h>
#include <stdint.h>

static PyObject *build_special_circuit(PyObject *self, PyObject *args) {
    QkCircuit *qc = qk_circuit_new(6, 0);

    // TODO 1: 큐비트 0에 H를 적용하세요.
    //   QkGate_H와 함께 qk_circuit_gate를 씁니다. C API 레퍼런스를 참고하세요.

    // TODO 2: 큐비트 1에 X를 적용하세요.
    //   QkGate_X와 함께 qk_circuit_gate를 씁니다.

    // TODO 3: CNOT 체인을 추가하세요. 제어는 큐비트 0, 대상은 1부터 5까지입니다.
    //   각 대상에 대해 QkGate_CX와 {제어, 대상} 배열로 qk_circuit_gate를 호출하세요.
    //   여기서는 for 루프가 쓸 만합니다.

    return qk_circuit_to_python_full(qc);
}

static PyMethodDef methods[] = {
    {"build_special_circuit", build_special_circuit, METH_NOARGS, "Build the special circuit."},
    {NULL, NULL, 0, NULL},
};
static struct PyModuleDef moduledef = {
    .m_base = PyModuleDef_HEAD_INIT,
    .m_name = "qgss_circuit",
    .m_methods = methods,
};
PyMODINIT_FUNC PyInit_qgss_circuit(void) {
    if (qk_import() < 0) {
        return NULL;
    }
    return PyModuleDef_Init(&moduledef);
}


TODO를 다 채웠으면 아래 셀을 실행해 확장을 빌드하세요.

In [ ]:
%%writefile setup.py
from setuptools import setup, Extension
import qiskit.capi
import sys
import subprocess

def supports_no_warn_duplicate():
    try:
        result = subprocess.run(
            ["ld", "-no_warn_duplicate_libraries"],
            capture_output=True
        )
        return b"unknown option" not in result.stderr
    except Exception:
        return False

extra_link_args = []
if sys.platform == "darwin" and supports_no_warn_duplicate():
    extra_link_args = ["-Wl,-no_warn_duplicate_libraries"]

setup(
    name="qgss_circuit",
    ext_modules=[
        Extension(
            name="qgss_circuit",
            sources=["qgss_circuit.c"],
            include_dirs=[qiskit.capi.get_include()],
            define_macros=[("QISKIT_PYTHON_EXTENSION", "1")],
            extra_link_args=extra_link_args,
        )
    ],
)


`setup.py build_ext --inplace`는 C 파일을 컴파일해 결과 모듈을 현재 디렉터리에 놓습니다. 그러면 바로 임포트할 수 있습니다.

In [ ]:
# C 파일을 Python 모듈로 빌드합니다.
subprocess.run([sys.executable, "setup.py", "build_ext", "--inplace"], check=True)
print("빌드 완료.")
print("이 셀을 전에 실행한 적이 있다면, 임포트하기 전에 커널을 재시작하세요 (Kernel > Restart).")

## 3.3 Python에서 상태 읽기

Qiskit C API는 회로를 만들 뿐 시뮬레이션은 하지 않습니다. 상태벡터를 얻으려면 회로를 Python으로 되돌린 뒤, 2장에서 쓴 도구를 그대로 쓰세요.

In [ ]:
import qgss_circuit

special_qc_c = qgss_circuit.build_special_circuit()
sv = Statevector(special_qc_c)
array_to_latex(sv, max_size=64)

In [ ]:
plot_state_qsphere(sv, show_state_labels=True)

2장에서 본 것과 같은 회로를 만들었으니, 회로를 제대로 조립했다면 최종 상태벡터가 grader의 연습 4를 통과합니다.
통과하지 못해도 괜찮습니다. 대부분은 앞에서 이미 grader를 통과했으므로, 여기서 틀려도 최종 진행 상황 결과는 그대로입니다. 이건 __선택__ 연습이라는 점을 기억하세요.

In [ ]:
grade_lab0_ex4(sv)

In [ ]:
check_progress("lab0")

# 추가 정보

**작성자:** Sophy Shin

**한글화:** 신소영 (Sophy)

**버전:** 1.0.0